# Chapter 7 &mdash; Simulating an NFA: Track the Set of Token Positions

**Concept 5 of the Chapter 7 decomposition:** *Simulating an NFA Without $\varepsilon$: Tracking the Set of Token Positions*

Follow the set of states the tokens occupy; accept when that set meets $F$ and the input is exhausted.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter7/Concept-Simulating-Without-Epsilon/Concept-Simulating-Without-Epsilon.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.AnimateNFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


Simulating an NFA is **not** backtracking. Keep a **set** of current states &mdash; where
all live tokens are &mdash; and on each symbol replace it by the union of the $\delta$
images.

$$\text{cur} \ \leftarrow\ \bigcup_{q\in\text{cur}} \delta(q,a)$$

Accept when the input is exhausted and $\text{cur} \cap F \neq \emptyset$.

The cost is $O(|Q|)$ memory and $O(|Q|^2)$ per symbol, **whatever the nondeterminism**
&mdash; and this simulation *is* the subset construction, performed lazily.

## 2. Definitions

### The machine

In [ ]:
N = md2mc('''NFA
I : 0 | 1 -> I
I : 1 -> A
A : 0 | 1 -> B
B : 0 | 1 -> F
''')

### The set simulation, written out

In [ ]:
def sim(N, s, show=False):
    cur = set(N["Q0"])
    if show: print("start : %s" % sorted(cur))
    for ch in s:
        cur = {t for q in cur for t in step_nfa(N, q, ch)}
        if show: print("on %s : %s" % (ch, sorted(cur)))
    return bool(cur & N["F"])

## 3. Tests

The set never exceeds $|Q|$, however many 'paths' there notionally are.

In [ ]:
ok = sim(N, '10101', show=True)
print("\naccepted?", ok, " accepts_nfa says", accepts_nfa(N, '10101'))
assert ok == accepts_nfa(N, '10101')

It agrees with Jove on everything short.

In [ ]:
from itertools import product
strs = [''.join(p) for k in range(11) for p in product('01', repeat=k)]
assert all(sim(N, s) == accepts_nfa(N, s) for s in strs)
print("set simulation matches accepts_nfa on all %d strings up to length 10" % len(strs))

Memory is bounded by the state set &mdash; that is the whole point.

In [ ]:
import random
longs = ''.join(random.choice('01') for _ in range(4000))
cur, peak = set(N["Q0"]), 0
for ch in longs:
    cur = {t for q in cur for t in step_nfa(N, q, ch)}
    peak = max(peak, len(cur))
print("input length %d, largest token set ever seen : %d  (|Q| = %d)"
      % (len(longs), peak, len(N["Q"])))
assert peak <= len(N["Q"])

An empty set means every token died &mdash; and it stays empty.

In [ ]:
Dead = md2mc('''NFA
I : 0 -> F
''')
cur = {t for q in Dead["Q0"] for t in step_nfa(Dead, q, '0')}
print("after '0' :", sorted(cur))
cur = {t for q in cur for t in step_nfa(Dead, q, '0')}
print("after '00':", cur, " <- every token has died, and the set stays empty")
assert cur == set() and not accepts_nfa(Dead, '00')

## 4. Animation

The token set as a highlighted group of states, moving as one.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateNFA import *
AnimateNFA(N, FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Why is backtracking simulation exponentially worse than this?
2. Modify `sim` to also report the largest set it ever holds.
3. What is the relationship between `sim` and `nfa2dfa`?

In [ ]:
# Your work for the exercises above.